In [1]:
from cooltools.api.insulation import calculate_insulation_score, find_boundaries
import cooler
import bioframe
import os
import pandas as pd

In [2]:
cname = '/gladstone/engelhardt/lab/lchumpitaz/hi-c/mcools/4DNFIXP4QG5B.mcool'
clr = cooler.Cooler(cname+'::resolutions/1000')

# only the 50kb window is used downstream (boundary_strength_50000);
# at 1kb resolution the 100kb window multiplies cooltools' O(window^2)
# pixel loop and dominated the runtime
windows = [50000]

# cache the full insulation table: re-runs load it instead of recomputing
cache_path = '../outputs/notebooks/GM12878/insulation_1000.tsv.gz'
if os.path.exists(cache_path):
    print('loading cached insulation table:', cache_path)
    insulation = pd.read_csv(cache_path, sep='\t')
else:
    insulation = calculate_insulation_score(clr, window_bp=windows, verbose=True, nproc=16)
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    insulation.to_csv(cache_path, sep='\t', index=False)
    print('cached insulation table to:', cache_path)

loading cached insulation table: ../outputs/notebooks/GM12878/insulation_1000.tsv.gz


In [3]:
ins_boundaries = find_boundaries(insulation, min_frac_valid_pixels=.5)
ins_boundaries['is_boundary'] = (ins_boundaries['boundary_strength_50000'].values>.15).astype(int)

In [4]:
import pyBigWig, os
out_dir = '../outputs/notebooks/GM12878'
os.makedirs(out_dir, exist_ok=True)
bw_path = out_dir+'/insul_1000.boundaries.bw'
bw = pyBigWig.open(bw_path, 'w')
bw.addHeader([(c, int(l)) for c, l in clr.chromsizes.items()])
bw.addEntries(ins_boundaries.chrom.tolist(),
              ins_boundaries.start.astype(int).tolist(),
              ends=ins_boundaries.end.astype(int).tolist(),
              values=ins_boundaries.is_boundary.astype(float).tolist())
bw.close()
print('wrote', bw_path)

wrote ../outputs/notebooks/GM12878/insul_1000.boundaries.bw
